# Disaster Tweet Classification: RoBERTa Transformer Fine-Tuning

**Architecture:** RoBERTa Base (`roberta-base`, 125M Parameters)  
**Objective:** End-to-end transformer fine-tuning with HuggingFace, PyTorch AMP (Mixed Precision), AdamW optimizer with Linear Warmup, Class-Weighted CrossEntropy Loss, per-model artifact serialization, and comprehensive disaster classification diagnostics across 10 classes.

---
### Notebook Structure
1. **Dataset Ingestion & Environment Setup**
2. **RoBERTa Tokenization (BPE) & PyTorch Dataset Preparation**
3. **Pretrained Transformer Model Setup & Class-Weighted Loss Function**
4. **Fine-Tuning Engine (FP16 Mixed Precision, Warmup Scheduler, Early Stopping)**
5. **Per-Model Artifact Serialization (`models/<model_name>/` and `saved_models/`)**
6. **Comprehensive Metrics Comparison (`metrics_comparison.csv` and `models_metrics_comparison.png`)**

In [ ]:
import os
import sys
import json
import time
import random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Setup directories
DATA_DIR = Path("dataset")
RESULTS_DIR = Path("results/06_transformer_roberta")
MODELS_DIR = RESULTS_DIR / "saved_models"
PER_MODEL_DIR = RESULTS_DIR / "models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PER_MODEL_DIR, exist_ok=True)

# Device & Reproducibility
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Utilizing compute device: {device}")

train_path = DATA_DIR / "train_clean.parquet"
val_path = DATA_DIR / "validation_clean.parquet"
test_path = DATA_DIR / "test_clean.parquet"

# Sourcing dataset from Kaggle or Google Drive
KAGGLE_INPUT_DIR = Path("/kaggle/input/humaid-disaster-tweets-parquet")
if KAGGLE_INPUT_DIR.exists():
    print("[+] Sourcing dataset from Kaggle dataset input...")
    for split in ["train", "validation", "test"]:
        p_clean = KAGGLE_INPUT_DIR / f"{split}_clean.parquet"
        p_raw = KAGGLE_INPUT_DIR / f"{split}.parquet"
        target_p = DATA_DIR / f"{split}_clean.parquet"
        if not target_p.exists():
            if p_clean.exists():
                pd.read_parquet(p_clean).to_parquet(target_p)
            elif p_raw.exists():
                pd.read_parquet(p_raw).to_parquet(target_p)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    raw_train = DATA_DIR / "train.parquet"
    raw_val = DATA_DIR / "validation.parquet"
    raw_test = DATA_DIR / "test.parquet"
    if not (raw_train.exists() and raw_val.exists() and raw_test.exists()):
        print("[+] Downloading HumAID dataset from Google Drive...")
        import gdown
        GDRIVE_URL = "https://drive.google.com/drive/folders/1pyMBc4SFc-sQvfmReiywPoN5cQbMpQBR?usp=drive_link"
        gdown.download_folder(url=GDRIVE_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)

train_file = train_path if train_path.exists() else DATA_DIR / "train.parquet"
val_file = val_path if val_path.exists() else DATA_DIR / "validation.parquet"
test_file = test_path if test_path.exists() else DATA_DIR / "test.parquet"

train_df = pd.read_parquet(train_file)
val_df = pd.read_parquet(val_file)
test_df = pd.read_parquet(test_file)

text_col = "clean_text" if "clean_text" in train_df.columns else "tweet_text"
print(f"[+] Loaded splits using column '{text_col}':")
print(f"    Train: {len(train_df):,} samples | Val: {len(val_df):,} samples | Test: {len(test_df):,} samples")

## 1. RoBERTa Tokenization & Dataset Preparation

In [ ]:
MODEL_CHECKPOINT = "roberta-base"
print(f"[+] Loading Tokenizer for {MODEL_CHECKPOINT}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# Encode labels
class_names = sorted(train_df["class_label"].unique())
label2idx = {name: i for i, name in enumerate(class_names)}
idx2label = {i: name for i, name in enumerate(class_names)}

y_train = train_df["class_label"].map(label2idx).values
y_val = val_df["class_label"].map(label2idx).values
y_test = test_df["class_label"].map(label2idx).values

MAX_LEN = 128
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TransformerDataset(train_df[text_col], y_train, tokenizer)
val_dataset = TransformerDataset(val_df[text_col], y_val, tokenizer)
test_dataset = TransformerDataset(test_df[text_col], y_test, tokenizer)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"[+] Prepared PyTorch DataLoaders with batch size = {BATCH_SIZE}.")

## 2. Model Initialization & Class-Weighted Loss

In [ ]:
print(f"[+] Initializing Pretrained Model: {MODEL_CHECKPOINT}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(class_names)
).to(device)

# Compute Class Weights for Loss
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

EPOCHS = 4
LEARNING_RATE = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = GradScaler()
print(f"[+] Configured AdamW with lr={LEARNING_RATE}, total_steps={total_steps}, warmup_steps={warmup_steps}.")

## 3. Fine-Tuning Loop with FP16 Mixed Precision & Early Stopping

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
best_val_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    # Training Phase
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation Phase
    model.eval()
    total_val_loss = 0.0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits, labels)
                
            total_val_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_macro_f1 = float(f1_score(val_targets, val_preds, average="macro", zero_division=0))
    val_acc = float(accuracy_score(val_targets, val_preds))
    
    history["train_loss"].append(float(avg_train_loss))
    history["val_loss"].append(float(avg_val_loss))
    history["val_macro_f1"].append(float(val_macro_f1))
    
    elapsed = time.time() - start_time
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{elapsed:.1f}s] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f} | Val Acc: {val_acc:.4f}")
    
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        print(f"  [+] New Best Val Macro F1 ({best_val_f1:.4f})! Saving checkpoint to saved_models...")
        model.save_pretrained(MODELS_DIR)
        tokenizer.save_pretrained(MODELS_DIR)

## 4. Final Evaluation & Per-Model Artifact Suite

In [ ]:
display_name_map = {
    'caution_and_advice': 'Caution & Advice',
    'displaced_people_and_evacuations': 'Displaced / Evac.',
    'infrastructure_and_utility_damage': 'Infrastructure',
    'injured_or_dead_people': 'Injured / Dead',
    'missing_or_found_people': 'Missing / Found',
    'not_humanitarian': 'Not Humanitarian',
    'other_relevant_information': 'Other Info',
    'requests_or_urgent_needs': 'Requests / Urgent',
    'rescue_volunteering_or_donation_effort': 'Rescue / Donation',
    'sympathy_and_support': 'Sympathy / Support'
}
display_names = [display_name_map.get(c, c) for c in class_names]

print("[+] Loading best model checkpoint for final Test Set evaluation...")
best_model = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR).to(device)
best_model.eval()

test_preds, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        with autocast():
            outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(labels.cpu().numpy())

# Compute Metrics
test_acc = float(accuracy_score(test_targets, test_preds))
test_f1_macro = float(f1_score(test_targets, test_preds, average='macro', zero_division=0))
test_f1_weighted = float(f1_score(test_targets, test_preds, average='weighted', zero_division=0))
test_prec_macro = float(precision_score(test_targets, test_preds, average='macro', zero_division=0))
test_rec_macro = float(recall_score(test_targets, test_preds, average='macro', zero_division=0))

model_name = "RoBERTa Base"
model_slug = "roberta_base"
curr_model_dir = PER_MODEL_DIR / model_slug
os.makedirs(curr_model_dir, exist_ok=True)

metrics_dict = {
    "Model": model_name,
    "Architecture": "Transformer (RoBERTa)",
    "Parameters": "125M",
    "Val Best Macro F1": float(best_val_f1),
    "Test Accuracy": test_acc,
    "Test Macro F1": test_f1_macro,
    "Test Weighted F1": test_f1_weighted,
    "Test Macro Precision": test_prec_macro,
    "Test Macro Recall": test_rec_macro,
}

# 1. Classification Report
report_str = classification_report(test_targets, test_preds, target_names=class_names, digits=4)
with open(curr_model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(f"=== Classification Report: {model_name} ===\n\n")
    f.write(report_str)

# 2. Metrics JSON
with open(curr_model_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_dict, f, indent=2)

# 3. Dual Confusion Matrix Plot
cm_raw = confusion_matrix(test_targets, test_preds)
cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), dpi=300)
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=display_names, yticklabels=display_names, ax=ax1)
ax1.set_title(f"Raw Confusion Matrix: {model_name}", fontsize=11, fontweight='bold')
ax1.set_xlabel("Predicted Label")
ax1.set_ylabel("True Label")
ax1.tick_params(axis='x', rotation=35, ha='right')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=display_names, yticklabels=display_names, ax=ax2)
ax2.set_title(f"Normalized Confusion Matrix: {model_name}", fontsize=11, fontweight='bold')
ax2.set_xlabel("Predicted Label")
ax2.set_ylabel("True Label")
ax2.tick_params(axis='x', rotation=35, ha='right')

plt.tight_layout()
plt.savefig(curr_model_dir / "confusion_matrix.png", bbox_inches='tight')
plt.close()

# 4. Per-Class Precision, Recall, and F1 Bar Chart
report_dict = classification_report(test_targets, test_preds, target_names=class_names, output_dict=True)
per_class_df = pd.DataFrame([
    {
        "Class": cls,
        "Precision": report_dict[cls]["precision"],
        "Recall": report_dict[cls]["recall"],
        "F1-Score": report_dict[cls]["f1-score"],
        "Support": report_dict[cls]["support"]
    }
    for cls in class_names
])
per_class_df.to_csv(curr_model_dir / "per_class_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
x = np.arange(len(class_names))
width = 0.25
ax.barh(x - width, per_class_df["Precision"], width, label="Precision", color="#3498db")
ax.barh(x, per_class_df["Recall"], width, label="Recall", color="#2ecc71")
ax.barh(x + width, per_class_df["F1-Score"], width, label="F1-Score", color="#e74c3c")
ax.set_yticks(x)
ax.set_yticklabels(class_names, fontsize=9)
ax.set_xlabel("Score", fontsize=10, fontweight='bold')
ax.set_title(f"Per-Class Performance: {model_name}", fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig(curr_model_dir / "per_class_metrics.png", bbox_inches='tight')
plt.close()

# 5. Training Curves Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
epochs_range = range(1, len(history["train_loss"]) + 1)
ax1.plot(epochs_range, history["train_loss"], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history["val_loss"], 'r-o', label='Val Loss')
ax1.set_title(f"Loss Progression: {model_name}", fontsize=11, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(epochs_range, history["val_macro_f1"], 'g-o', label='Val Macro F1')
ax2.set_title(f"Val Macro F1 Progression: {model_name}", fontsize=11, fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Macro F1")
ax2.legend()

plt.tight_layout()
plt.savefig(curr_model_dir / "training_curves.png", bbox_inches='tight')
plt.close()

# 6. Save Comparison Metrics & Plot at Notebook Level
summary_df = pd.DataFrame([metrics_dict])
summary_df.to_csv(RESULTS_DIR / "metrics_comparison.csv", index=False)
print("\n=== RoBERTa Fine-Tuning Performance Summary ===")
print(summary_df.to_string(index=False))

# Plot Comparison Figure of All Models in this Notebook (Test Macro F1 vs Accuracy)
df_sorted = summary_df.sort_values(by="Test Macro F1", ascending=False).reset_index(drop=True)

model_labels = [m for m in df_sorted["Model"]]
x = np.arange(len(df_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=1)

bars1 = ax.bar(x - width/2, df_sorted["Test Macro F1"], width, label="Macro F1", color="#1f77b4", edgecolor="none", zorder=3)
bars2 = ax.bar(x + width/2, df_sorted["Test Accuracy"], width, label="Accuracy", color="#ff7f0e", edgecolor="none", zorder=3)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f"{height:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8.5)
                
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f"{height:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8.5)

ax.set_title("RoBERTa Transformer Benchmark: Test Macro F1 vs Accuracy", fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel("Model", fontsize=11, fontweight='bold', labelpad=10)
ax.set_ylabel("Score", fontsize=11, fontweight='bold', labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=35, ha='right', fontsize=9.5)

max_val = max(df_sorted["Test Macro F1"].max(), df_sorted["Test Accuracy"].max())
ax.set_ylim(0, min(1.0, max_val + 0.12))
ax.legend(loc="upper right", frameon=True, fontsize=9.5)

for spine in ax.spines.values():
    spine.set_color('#888888')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "models_metrics_comparison.png", bbox_inches='tight')
plt.show()

print(f"[+] All artifacts, models, reports, and comparison plots saved to: {RESULTS_DIR.resolve()}")